In [1]:
!pip install -q wandb protobuf tiktoken
# !pip install --upgrade -q jax flax
# !pip install --upgrade -q "jax[tpu]" flax
!pip install --upgrade -q "jax[cuda12]" flax

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 531.3/531.3 kB 13.5 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 67.3 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.2/8.2 MB 98.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.8/175.8 MB 10.4 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.3/87.3 MB 22.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 MB 50.8 MB/s eta 0:00:00:00:0100:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.


In [2]:
import os
# xla_dump_path = "/kaggle/working/xla_debug"
# os.makedirs(xla_dump_path, exist_ok=True)
# os.environ["XLA_FLAGS"] = (
#     f"--xla_dump_to={xla_dump_path} "
#     f"--xla_dump_hlo_as_text "
#     f"--xla_dump_hlo_snapshots "
#     f"--xla_dump_hlo_as_html"
# )

os.environ["XLA_FLAGS"] = "--xla_force_host_platform_device_count=8"

In [3]:
import jax
import jax.numpy as jnp
from jax.sharding import PartitionSpec as P
import jax.experimental.pallas as pl
from jax.numpy import einsum
from einops import rearrange
from flax import nnx
import flax
import orbax.checkpoint as ocp
import pandas as pd
import numpy as np
import optax
import math
import tqdm
import random
from datetime import datetime
import matplotlib.pyplot as plt
import grain
from collections.abc import Iterable
from typing import Literal, Union, Sequence, Optional, Any, Callable, Tuple, List
import cv2
import matplotlib.pyplot as plt
from ml_collections import ConfigDict
import functools
from flax import jax_utils
import wandb
import gc
import time
from types import SimpleNamespace
from collections import defaultdict
from IPython.display import Image
from IPython.core import ultratb
import sys
import tiktoken
import seaborn as sns

jax.config.update("jax_traceback_filtering", "off")
jax.config.update("jax_debug_nans", True)
sys.excepthook = ultratb.FormattedTB(mode='Verbose', call_pdb=False)

In [4]:
print(f"""
    jax -> {jax.__version__}
    flax -> {flax.__version__}
    orbax -> {ocp.__version__}
    grain -> {grain.__version__}
""")


    jax -> 0.11.0
    flax -> 0.12.8
    orbax -> 0.11.34
    grain -> 0.2.16



In [5]:
jax.devices()

[CudaDevice(id=0), CudaDevice(id=1)]

In [6]:
jax.devices()[0].platform

'gpu'

In [7]:
devices = np.array(jax.devices())
if devices.shape[0] == 2:
    devices = devices.reshape(2, 1)
elif devices.shape[0] == 8:
    devices = devices.reshape(4, 2)
else:
    devices = devices[:, np.newaxis]

MESH = jax.sharding.Mesh(
    devices=devices,
    axis_names=("x", "y"),
)

vmap_devices = np.array(jax.devices())
VMAP_MESH = jax.sharding.Mesh(
    devices=vmap_devices,
    axis_names=("x",),
)
nnx.display(MESH, VMAP_MESH)

## Config Definition

In [8]:
tokenizer = tiktoken.get_encoding("gpt2")
tokenizer._special_tokens

{'<|endoftext|>': 50256}

In [9]:
def get_config():
    config = ConfigDict()
    
    data = ConfigDict()
    # data.batch_size = 144 * 4
    data.batch_size = 120
    data.vocab_size = tokenizer.n_vocab

    model = ConfigDict()
    model.seed = 2635276
    model.max_len = 257
    model.d = 256
    model.dropout_rate = 0.1
    model.num_heads = 8
    model.num_resids = 4
    model.head_dim = model.d // model.num_heads
    model.ff_expan = 4
    model.ln_epsilon = 1e-6
    model.q_chunk_size = 64
    model.k_chunk_size = 64

    training = ConfigDict()
    training.lr = 1e-3
    training.epochs = 1
    training.save_and_sample_every = 1000

    sampling = ConfigDict()
    sampling.temperature = 0.95
    sampling.top_k = 10
    sampling.batch_size = 16
    
    config.data = data
    config.model = model
    config.training = training
    config.sampling = sampling
    
    return config

config = get_config()

## Initialize W&B

In [10]:
# Optional W&B (set WANDB_API_KEY env var — never commit keys)
# run = wandb.init(project="jax-flash-minigpt", config=config.to_dict())
run = None

## Dataset Pipeline

In [11]:
BASE_DIR = "/kaggle/input/datasets/thedevastator/tinystories-narrative-classification"
TRAIN_CSV = f"{BASE_DIR}/train.csv"
VAL_CSV = f"{BASE_DIR}/validation.csv"

train_df = pd.read_csv(TRAIN_CSV)
val_df = pd.read_csv(VAL_CSV)

print(train_df.shape, val_df.shape)

train_stories = [f"{ele.strip()}<|endoftext|>" for ele in train_df["text"].dropna().tolist()]
val_stories = [f"{ele.strip()}<|endoftext|>" for ele in val_df["text"].dropna().tolist()]

print(len(train_stories), len(val_stories))

(2119719, 1) (21990, 1)
2119489 21990


In [12]:
# lengths = jnp.array(
#     [len(ele1) for ele1 in train_stories] + [len(ele2) for ele2 in val_stories],
#     dtype=jnp.int32
# )
# sns.histplot(lengths, kde=True)
# sns.histplot(lengths, kde=True, cumulative=True)

In [13]:
class DataSource(grain.sources.RandomAccessDataSource):
    def __init__(self, stories):
        self.stories = stories
        
    def __getitem__(self, idx):
        return self.stories[idx]
        
    def __len__(self):
        return len(self.stories)

In [14]:
def load_story(story: str):
    encoding = tokenizer.encode(
        story,
        allowed_special={"<|endoftext|>"}
    )[:config.model.max_len]
    encoding = encoding + [0] * (config.model.max_len - len(encoding))

    inputs = jnp.array(encoding[:-1], dtype=jnp.int32)
    targets = jnp.array(encoding[1:], dtype=jnp.int32)
    return inputs, targets

def shard_stories(x):
    sharded_inputs = jax.device_put(
        x[0],
        jax.sharding.NamedSharding(
            MESH,
            P(
                "x" if x[0].shape[0] % devices.shape[0] == 0 else None,
                "y" if x[0].shape[1] % devices.shape[1] == 0 else None,
            )                             
        )
    )

    sharded_targets = jax.device_put(
        x[1],
        jax.sharding.NamedSharding(
            MESH,
            P(
                "x" if x[1].shape[0] % devices.shape[0] == 0 else None,
                "y" if x[1].shape[1] % devices.shape[1] == 0 else None,)
        )
    )

    return sharded_inputs, sharded_targets

print(f"BATCH_SIZE - {config.data.batch_size}")

train_source = DataSource(train_stories)    
train_dataset = (
    grain.MapDataset.source(train_source)
    .shuffle(seed=677456347)
    .map(load_story)
    .batch(batch_size=config.data.batch_size, drop_remainder = True)
    .map(shard_stories)
)

val_source = DataSource(val_stories)
val_dataset = (
    grain.MapDataset.source(val_source)
    .shuffle(seed=677456347)
    .map(load_story)
    .batch(batch_size=config.sampling.batch_size, drop_remainder = True)
    .map(shard_stories)
)

print(len(train_dataset), len(val_dataset))

BATCH_SIZE - 120
17662 1374


In [15]:
for index, batch in enumerate(train_dataset):    
    nnx.display(batch)
    
    if index == 0:
        break

## Initializers

In [16]:
kernel_init_fn = nnx.initializers.xavier_uniform()
embed_init_fn = nnx.initializers.variance_scaling(1.0, 'fan_in', 'normal', out_axis=0)
scale_init_fn = nnx.initializers.ones_init()
bias_init_fn = nnx.initializers.zeros_init()

## Half Precision Utility Function

## Embedding

In [17]:
class Embeddings(nnx.Module):
    def __init__(self, rngs: nnx.Rngs):
        self.emb_inputs = nnx.Embed(
            num_embeddings = config.data.vocab_size,
            features = config.model.d,
            embedding_init=nnx.with_partitioning(
                embed_init_fn,
                ("x" if config.data.vocab_size % devices.shape[0] == 0 else None, None),
                mesh = MESH,
            ),
            rngs = rngs
        )

        self.emb_pos = nnx.Embed(
            num_embeddings = config.model.max_len,
            features = config.model.d,
            embedding_init=nnx.with_partitioning(
                embed_init_fn,
                ("x" if config.model.max_len % devices.shape[0] == 0 else None, None),
                mesh = MESH
            ),
            rngs = rngs
        )
            
    def __call__(self, x: jnp.ndarray, position_offset=0):
        B, T = x.shape
        
        token_emb = self.emb_inputs(x)
        
        positions = position_offset + jnp.arange(0, x.shape[1])[None, :]
        pos_emb = self.emb_pos(positions)
        
        return token_emb + pos_emb

## KV Cache

In [18]:
class KVCache(nnx.Module):
    def __init__(
        self,
        batch_size: int,
        num_heads: int,
        max_seq_len: int,
        d: int,
    ):
        self.k = nnx.Variable(
            jnp.zeros((batch_size * num_heads, max_seq_len, d), dtype=jnp.float32),
            sharding_names = ("x" if (batch_size * num_heads) % devices.shape[0] == 0 else None, None, None),
            mesh = VMAP_MESH
        )
        self.v = nnx.Variable(
            jnp.zeros((batch_size * num_heads, max_seq_len, d), dtype=jnp.float32),
            sharding_names = ("x" if batch_size % devices.shape[0] == 0 else None, None, None),
            mesh = VMAP_MESH
        )
        self.pos = nnx.Variable(jnp.array(0, dtype=jnp.int32))

    def reset(self):
        self.k[...] = jax.device_put(
            jnp.zeros_like(self.k[...], dtype=jnp.float32),
            jax.sharding.NamedSharding(MESH, P("x" if self.k[...].shape[0] % devices.shape[0] == 0 else None, None, None))
        )
        self.v[...] = jax.device_put(
            jnp.zeros_like(self.v[...], dtype=jnp.float32),
            jax.sharding.NamedSharding(MESH, P("x" if self.v[...].shape[0] % devices.shape[0] == 0 else None, None, None))
        )
        self.pos[...] = jnp.array(0, dtype=jnp.int32)

In [19]:
# NEG_INF = -1e6
NEG_INF = jnp.finfo(jnp.float32).min
NEG_INF

np.float32(-3.4028235e+38)

## Flash Attention

In [71]:
class FlashAttention(nnx.Module):
    def __init__(self, rngs: nnx.Rngs):
        self.Whqkv = nnx.Linear(
            in_features = config.model.d,
            out_features = 3 * config.model.d,
            use_bias = False,
            kernel_init=nnx.with_partitioning(
                kernel_init_fn,
                (
                    "x" if config.model.d % devices.shape[0] == 0 else None,
                    "y" if (3 * config.model.d) % devices.shape[1] == 0 else None,
                ),
                mesh = MESH
            ),
            bias_init=None, 
            rngs = rngs
        )

        self.Wp = nnx.Linear(
            in_features = config.model.d,
            out_features = config.model.d,
            use_bias=False,
            kernel_init=nnx.with_partitioning(
                kernel_init_fn,
                (
                    "x" if config.model.d % devices.shape[0] == 0 else None,
                    "y" if config.model.d % devices.shape[1] == 0 else None,
                ),
                mesh = MESH
            ),
            bias_init=None, 
            rngs = rngs
        )

        self.cache = KVCache(
            batch_size = config.sampling.batch_size,
            num_heads = config.model.num_heads,
            max_seq_len = config.model.max_len,
            d = config.model.head_dim
        )

    def __call__(self, x: jnp.ndarray):
        B, T = x.shape[:2]
        D = config.model.head_dim
        H = config.model.num_heads
        
        hqkvd = self.Whqkv(x)
        qkv = jnp.permute_dims(hqkvd.reshape(B, T, 3, H, D), axes=(0, 3, 1, 2, 4)) # (B, H, T, 3, D)
        
        #-------------------------------------------------------------------
        q = jax.lax.with_sharding_constraint(
            jax.lax.dynamic_index_in_dim(qkv, 0, axis=3, keepdims=False).reshape(B * H, T, D),
            jax.sharding.NamedSharding(VMAP_MESH, P("x" if B % devices.shape[0] == 0 else None, None, None))
        ) # (B * H, T, D)
        k = jax.lax.with_sharding_constraint(
            jax.lax.dynamic_index_in_dim(qkv, 1, axis=3, keepdims=False).reshape(B * H, T, D),
            jax.sharding.NamedSharding(VMAP_MESH, P("x" if B % devices.shape[0] == 0 else None, None, None))
        ) # (B * H, T, D)
        v = jax.lax.with_sharding_constraint(
            jax.lax.dynamic_index_in_dim(qkv, 2, axis=3, keepdims=False).reshape(B * H, T, D),
            jax.sharding.NamedSharding(VMAP_MESH, P("x" if B % devices.shape[0] == 0 else None, None, None))
        ) # (B * H, T, D)
        #-------------------------------------------------------------------
        
        out = flash_attention(q, k, v).reshape(B, H, T, D) # (B, H, T, D)
        out = jnp.permute_dims(out, (0, 2, 1, 3)).reshape(B, T, H * D)
        return self.Wp(out)

    #PREPARE BLOCK ---------------------------------------------------------------------------------------------
    def prepare(self, x: jnp.ndarray):
        B, T = x.shape[:2]
        D = config.model.head_dim
        H = config.model.num_heads

        self.cache.reset()
        
        hqkvd = self.Whqkv(x)
        qkv = jnp.permute_dims(hqkvd.reshape(B, T, 3, H, D), axes=(0, 3, 1, 2, 4)) # (B, H, T, 3, D)

        #-------------------------------------------------------------------
        q = jax.lax.with_sharding_constraint(
            jax.lax.dynamic_index_in_dim(qkv, 0, axis=3, keepdims=False).reshape(B * H, T, D),
            jax.sharding.NamedSharding(VMAP_MESH, P("x" if B % devices.shape[0] == 0 else None, None, None))
        ) # (B * H, T, D)
        k = jax.lax.with_sharding_constraint(
            jax.lax.dynamic_index_in_dim(qkv, 1, axis=3, keepdims=False).reshape(B * H, T, D),
            jax.sharding.NamedSharding(VMAP_MESH, P("x" if B % devices.shape[0] == 0 else None, None, None))
        ) # (B * H, T, D)
        v = jax.lax.with_sharding_constraint(
            jax.lax.dynamic_index_in_dim(qkv, 2, axis=3, keepdims=False).reshape(B * H, T, D),
            jax.sharding.NamedSharding(VMAP_MESH, P("x" if B % devices.shape[0] == 0 else None, None, None))
        ) # (B * H, T, D)
        #-------------------------------------------------------------------
        self.cache.k[...] = jax.lax.dynamic_update_slice_in_dim(self.cache.k[...], k, 0, axis=1)
        self.cache.v[...] = jax.lax.dynamic_update_slice_in_dim(self.cache.v[...], v, 0, axis=1)

        out = flash_attention(q, k, v).reshape(B, H, T, D) # (B, H, T, D)
        out = jnp.permute_dims(out, axes = (0, 2, 1, 3)).reshape(B, T, H * D) # (B, T, H * D)
        out = self.Wp(out) # (B, T, D)

        self.cache.pos[...] = T
        return out # (B, T, D)

    #INFERENCE BLOCK-----------------------------------------------------------------------------------------------
    def inference(self, x: jnp.ndarray):
        B, T = x.shape[:2]
        assert T == 1, "Inference x should have only one timestamp"
        
        D = config.model.head_dim
        H = config.model.num_heads

        max_seq_len = self.cache.k[...].shape[2]
        current_pos = self.cache.pos[...]
        
        hqkvd_new = self.Whqkv(x)
        qkv_new = jnp.permute_dims(hqkvd_new.reshape(B, T, 3, H, D), axes=(0, 3, 1, 2, 4)) # (B, H, 1, 3, D)

        #-------------------------------------------------------------------
        q_new = jax.lax.with_sharding_constraint(
            jax.lax.dynamic_index_in_dim(qkv_new, 0, axis=3, keepdims=False).reshape(B * H, T, D),
            jax.sharding.NamedSharding(VMAP_MESH, P("x" if B % devices.shape[0] == 0 else None, None, None))
        ) # (B * H, 1, D)
        k_new = jax.lax.with_sharding_constraint(
            jax.lax.dynamic_index_in_dim(qkv_new, 1, axis=3, keepdims=False).reshape(B * H, T, D),
            jax.sharding.NamedSharding(VMAP_MESH, P("x" if B % devices.shape[0] == 0 else None, None, None))
        ) # (B * H, 1, D)
        v_new = jax.lax.with_sharding_constraint(
            jax.lax.dynamic_index_in_dim(qkv_new, 2, axis=3, keepdims=False).reshape(B * H, T, D),
            jax.sharding.NamedSharding(VMAP_MESH, P("x" if B % devices.shape[0] == 0 else None, None, None))
        )# (B * H, 1, D)
        #-------------------------------------------------------------------
        self.cache.k[...] = jax.lax.dynamic_update_slice_in_dim(self.cache.k[...], k_new, current_pos, axis=1)
        self.cache.v[...] = jax.lax.dynamic_update_slice_in_dim(self.cache.v[...], v_new, current_pos, axis=1)
        self.cache.pos[...] = current_pos + 1

        k_full = self.cache.k[...]  # (B * H, T, D)
        v_full = self.cache.v[...]   # (B * H ,T, D)
        k_T_full = jnp.transpose(k_full, axes=(0, 2, 1)) # (B * H, D, T)
        
        scores = (q_new @ k_T_full) / math.sqrt(D) #Need to offload this to each head # (B * H, 1, T)
        mask = jnp.arange(k_full.shape[1]) < (current_pos + 1)
        scores = jnp.where(mask[None, None, :], scores, NEG_INF)
        
        attn = jax.nn.softmax(scores, axis = -1) # (B * H, 1, T)
        out = (attn @ v_full).reshape(B, H, T, D) # (B * H, 1, T) @ (B * H, T, D) -> (B * H, 1, D) -> (B, H, 1, D)
        out = jnp.permute_dims(out, axes = (0, 2, 1, 3)).reshape(B, T, H * D) # (B, 1, H * D)
        out = self.Wp(out) # (B, 1, D)
        return out # (B, 1, D)

## Flash Forward

In [21]:
def get_attn_mask(
    q_idx, k_idx,
    Q_BLOCK, K_BLOCK,
    Q_SIZE, K_SIZE,
    BH, T, D
):
    
    x = q_idx * Q_BLOCK + jnp.arange(Q_SIZE, dtype = jnp.int32)[:, None] # (Q, 1)
    y = k_idx * K_BLOCK + jnp.arange(K_SIZE, dtype = jnp.int32)[None, :] # (1, K)
    
    basic_mask = x >= y # (Q, K)
    return basic_mask[None, ...]

def forward_kernel(
    q_ref, k_T_ref, v_ref, o_ref, m_ref, l_ref,
    Q_BLOCK, K_BLOCK, BH, T, D, get_attn_mask
):  
    for q_idx in range((T + Q_BLOCK - 1) // Q_BLOCK):
        q = q_ref[..., q_idx * Q_BLOCK : (q_idx + 1) * Q_BLOCK, :]
    
        mi = jnp.full((BH, min(Q_BLOCK, q.shape[1]), 1), NEG_INF, dtype=jnp.float32)
        li = jnp.zeros((BH, min(Q_BLOCK, q.shape[1]), 1), dtype=jnp.float32)
        o = jnp.zeros((BH, min(Q_BLOCK, q.shape[1]), D), dtype=jnp.float32)
    
        for k_idx in range((T + K_BLOCK - 1) // K_BLOCK):
            k_T = k_T_ref[..., k_idx * K_BLOCK : (k_idx + 1) * K_BLOCK]
            v = v_ref[..., k_idx * K_BLOCK : (k_idx + 1) * K_BLOCK, :]

            # k_T = jnp.transpose(k, (0, 2, 1))
            scores = (q @ k_T) / jnp.sqrt(D)
            attn_mask = get_attn_mask(
                q_idx, k_idx,
                Q_BLOCK, K_BLOCK,
                min(Q_BLOCK, q.shape[1]),
                min(K_BLOCK, k_T.shape[2]),
                BH, T, D
            )
            scores += jnp.where(attn_mask, 0.0, NEG_INF)

            mij = jnp.max(scores, axis=-1, keepdims=True)
            pij = jnp.exp(scores - mij)
            lij = jnp.sum(pij, axis=-1, keepdims=True)
            
            mi_new = jnp.maximum(mi, mij)
            alpha = jnp.exp(mi - mi_new)
            beta = jnp.exp(mij - mi_new)
            li_new = alpha * li + beta * lij

            #scale p
            pij_scale = beta / li_new
            pij = pij * pij_scale
            #scale o
            o_scale = (li / li_new) * alpha
            o *= o_scale
            #update acc
            o += pij @ v
    
            # o = (li * alpha * o + beta * (pij @ v)) / li_new
            
            li = li_new
            mi = mi_new
        
        o_ref[..., q_idx * Q_BLOCK : (q_idx + 1) * Q_BLOCK, :] = o
        m_ref[..., q_idx * Q_BLOCK : (q_idx + 1) * Q_BLOCK, :] = mi
        l_ref[..., q_idx * Q_BLOCK : (q_idx + 1) * Q_BLOCK, :] = li

def flash_forward(q, k, v):
    BH, T, D = q.shape

    Q_BLOCK = config.model.q_chunk_size
    K_BLOCK = config.model.k_chunk_size

    BH = BH // vmap_devices.shape[0]

    k_T = jnp.transpose(k, (0, 2, 1))
    v_T = jnp.transpose(v, (0, 2, 1))

    pforward = pl.pallas_call(
        functools.partial(
            forward_kernel,
            Q_BLOCK=Q_BLOCK,
            K_BLOCK=K_BLOCK,
            BH=BH, T=T, D=D,
            get_attn_mask=get_attn_mask,
        ),
        out_shape=[
            jax.ShapeDtypeStruct((BH, T, D), jnp.float32),
            jax.ShapeDtypeStruct((BH, T, 1), jnp.float32),
            jax.ShapeDtypeStruct((BH, T, 1), jnp.float32)
        ],
        grid=(1,), 
        in_specs=[
            pl.BlockSpec(block_shape=(BH, T, D), index_map=lambda i: (0, 0, 0)),
            pl.BlockSpec(block_shape=(BH, D, T), index_map=lambda i: (0, 0, 0)),
            pl.BlockSpec(block_shape=(BH, T, D), index_map=lambda i: (0, 0, 0)),
        ],
        out_specs=[
            pl.BlockSpec(
                block_shape=(BH, T, D),
                index_map=lambda i: (0, 0, 0),
            ),
            pl.BlockSpec(
                block_shape=(BH, T, 1),
                index_map=lambda i: (0, 0, 0),
            ),
            pl.BlockSpec(
                block_shape=(BH, T, 1),
                index_map=lambda i: (0, 0, 0),
            ),
        ],
        interpret=True,
    )

    o, m, l = jax.jit(
        jax.shard_map(
            pforward,
            mesh=VMAP_MESH,
            in_specs=(
                P("x", None, None),
                P("x", None, None),
                P("x", None, None),
            ),
            out_specs=(
                P("x", None, None),
                P("x", None, None),
                P("x", None, None)
            ),
            check_vma=False
        )
    )(q, k_T, v)
    
    return o, (q, k, k_T, v_T, o, m, l)

## Flash Backward

In [22]:
def backward_kernel_q(
    q_ref, k_ref, k_T_ref, v_T_ref, o_ref, do_ref, m_ref, l_ref, dq_ref, pij_ref, dscores_ref,
    Q_BLOCK, K_BLOCK, BH, T, D, get_attn_mask
):
    for q_idx in range((T + Q_BLOCK - 1) // Q_BLOCK):
        q = q_ref[..., q_idx * Q_BLOCK : (q_idx + 1) * Q_BLOCK, :]
        o = o_ref[..., q_idx * Q_BLOCK : (q_idx + 1) * Q_BLOCK, :]
        do = do_ref[..., q_idx * Q_BLOCK : (q_idx + 1) * Q_BLOCK, :]
        mi = m_ref[..., q_idx * Q_BLOCK : (q_idx + 1) * Q_BLOCK, :]
        li = l_ref[..., q_idx * Q_BLOCK : (q_idx + 1) * Q_BLOCK, :]
    
        dq = jnp.zeros(q.shape, dtype=jnp.float32)
        for k_idx in range((T + K_BLOCK - 1) // K_BLOCK):
            k = k_ref[..., k_idx * K_BLOCK : (k_idx + 1) * K_BLOCK, :]
            k_T = k_T_ref[..., k_idx * K_BLOCK : (k_idx + 1) * K_BLOCK]
            v_T = v_T_ref[..., k_idx * K_BLOCK : (k_idx + 1) * K_BLOCK]

            scores = (q @ k_T) / jnp.sqrt(D)
            attn_mask = get_attn_mask(
                q_idx, k_idx,
                Q_BLOCK, K_BLOCK,
                min(Q_BLOCK, q.shape[1]),
                min(K_BLOCK, k.shape[1]),
                BH, T, D
            )
            scores += jnp.where(attn_mask, 0.0, NEG_INF)
            
            pij = jnp.exp(scores - mi) / li
            dpij = do @ v_T
            ddi = jnp.sum(do * o, axis = -1, keepdims=True)
            dscores = (pij * (dpij - ddi)) / jnp.sqrt(D)
            
            dq += dscores @ k

            pij_ref[..., q_idx * Q_BLOCK : (q_idx + 1) * Q_BLOCK, k_idx * K_BLOCK : (k_idx + 1) * K_BLOCK] = pij
            dscores_ref[..., q_idx * Q_BLOCK : (q_idx + 1) * Q_BLOCK, k_idx * K_BLOCK : (k_idx + 1) * K_BLOCK] = dscores
            
        dq_ref[..., q_idx * Q_BLOCK : (q_idx + 1) * Q_BLOCK, :] = dq

def backward_kernel_kv(
    q_ref, do_ref, pij_T_ref, dscores_T_ref, dk_ref, dv_ref,
    Q_BLOCK, K_BLOCK, BH, T, D, get_attn_mask
):
    for k_idx in range((T + K_BLOCK - 1) // K_BLOCK):
        dk = jnp.zeros((BH, min(K_BLOCK, T - k_idx * K_BLOCK), D), dtype=jnp.float32)
        dv = jnp.zeros((BH, min(K_BLOCK, T - k_idx * K_BLOCK), D), dtype=jnp.float32)
    
        for q_idx in range((T + Q_BLOCK - 1) // Q_BLOCK):
            q = q_ref[..., q_idx * Q_BLOCK : (q_idx + 1) * Q_BLOCK, :]
            do = do_ref[..., q_idx * Q_BLOCK : (q_idx + 1) * Q_BLOCK, :]
            pij_T =  pij_T_ref[..., k_idx * K_BLOCK : (k_idx + 1) * K_BLOCK, q_idx * Q_BLOCK : (q_idx + 1) * Q_BLOCK]
            dscores_T =  dscores_T_ref[..., k_idx * K_BLOCK : (k_idx + 1) * K_BLOCK, q_idx * Q_BLOCK : (q_idx + 1) * Q_BLOCK]
            
            dk += dscores_T @ q
            dv += pij_T @ do
        
        dk_ref[..., k_idx * K_BLOCK : (k_idx + 1) * K_BLOCK, :] = dk
        dv_ref[..., k_idx * K_BLOCK : (k_idx + 1) * K_BLOCK, :] = dv

def flash_backward(res, g):
    q, k, k_T, v_T, o, m, l = res
    do = g
    BH, T, D = q.shape
    
    Q_BLOCK = config.model.q_chunk_size
    K_BLOCK = config.model.k_chunk_size
    
    BH = BH // vmap_devices.shape[0]
    
    pbackward_q = pl.pallas_call(
        functools.partial(
            backward_kernel_q,
            Q_BLOCK=Q_BLOCK,
            K_BLOCK=K_BLOCK,
            BH=BH, T=T, D=D,
            get_attn_mask=get_attn_mask,
        ),
        out_shape=[
            jax.ShapeDtypeStruct((BH, T, D), jnp.float32),
            jax.ShapeDtypeStruct((BH, T, T), jnp.float32),
            jax.ShapeDtypeStruct((BH, T, T), jnp.float32),
        ],
        grid=(1,), 
        in_specs=[
            pl.BlockSpec(block_shape=(BH, T, D), index_map=lambda i: (0, 0, 0)), # q
            pl.BlockSpec(block_shape=(BH, T, D), index_map=lambda i: (0, 0, 0)), # k
            pl.BlockSpec(block_shape=(BH, D, T), index_map=lambda i: (0, 0, 0)), # k_T
            pl.BlockSpec(block_shape=(BH, D, T), index_map=lambda i: (0, 0, 0)), # v_T
            pl.BlockSpec(block_shape=(BH, T, D), index_map=lambda i: (0, 0, 0)), # o
            pl.BlockSpec(block_shape=(BH, T, D), index_map=lambda i: (0, 0, 0)), # do
            pl.BlockSpec(block_shape=(BH, T, 1), index_map=lambda i: (0, 0, 0)), # m
            pl.BlockSpec(block_shape=(BH, T, 1), index_map=lambda i: (0, 0, 0)), # l
        ],
        out_specs=[
            pl.BlockSpec(
                block_shape=(BH, T, D),
                index_map=lambda i: (0, 0, 0),
            ),
            pl.BlockSpec(
                block_shape=(BH, T, T),
                index_map=lambda i: (0, 0, 0),
            ),
            pl.BlockSpec(
                block_shape=(BH, T, T),
                index_map=lambda i: (0, 0, 0),
            ),
        ],
        interpret=True
    )

    pbackward_kv = pl.pallas_call(
        functools.partial(
            backward_kernel_kv,
            Q_BLOCK=Q_BLOCK,
            K_BLOCK=K_BLOCK,
            BH=BH, T=T, D=D,
            get_attn_mask=get_attn_mask,
        ),
        out_shape=[
            jax.ShapeDtypeStruct((BH, T, D), dtype=jnp.float32),
            jax.ShapeDtypeStruct((BH, T, D), dtype=jnp.float32)
        ],
        grid=(1,), 
        in_specs=[
            pl.BlockSpec(block_shape=(BH, T, D), index_map=lambda i: (0, 0, 0)), # q
            pl.BlockSpec(block_shape=(BH, T, D), index_map=lambda i: (0, 0, 0)), # do
            pl.BlockSpec(block_shape=(BH, T, T), index_map=lambda i: (0, 0, 0)), # pij
            pl.BlockSpec(block_shape=(BH, T, T), index_map=lambda i: (0, 0, 0)), # dscores
        ],
        out_specs=[
            pl.BlockSpec(
                block_shape=(BH, T, D),
                index_map=lambda i: (0, 0, 0),
            ),
            pl.BlockSpec(
                block_shape=(BH, T, D),
                index_map=lambda i: (0, 0, 0),
            ),
        ],
        interpret=True
    )

    dq, pij, dscores = jax.jit(
        jax.shard_map(
            pbackward_q,
            mesh=VMAP_MESH,
            in_specs=(
                P("x", None, None),
                P("x", None, None),
                P("x", None, None),
                P("x", None, None),
                P("x", None, None),
                P("x", None, None),
                P("x", None, None),
                P("x", None, None),
            ),
            out_specs=(
                P("x", None, None),
                P("x", None, None),
                P("x", None, None)
            ),
            check_vma=False
        )
    )(q, k, k_T, v_T, o, do, m, l)

    pij_T = jnp.transpose(pij, (0, 2, 1))
    dscores_T = jnp.transpose(dscores, (0, 2, 1))

    dk, dv = jax.jit(
        jax.shard_map(
            pbackward_kv,
            mesh=VMAP_MESH,
            in_specs=(
                P("x", None, None),
                P("x", None, None),
                P("x", None, None),
                P("x", None, None),
            ),
            out_specs=(
                P("x", None, None),
                P("x", None, None),
            ),
            check_vma=False
        )
    )(q, do, pij_T, dscores_T)
    return dq, dk, dv

In [23]:
q = jax.device_put(
    jnp.ones(shape = (4 * 8, 12, 16), dtype=jnp.float32),
    jax.sharding.NamedSharding(VMAP_MESH, P("x", None, None))
)
k = jax.device_put(
    jnp.ones(shape = (4 * 8, 12, 16), dtype=jnp.float32),
    jax.sharding.NamedSharding(VMAP_MESH, P("x", None, None))
)
v = jax.device_put(
    jnp.ones(shape = (4 * 8, 12, 16), dtype=jnp.float32),
    jax.sharding.NamedSharding(VMAP_MESH, P("x", None, None))
)

o, res = flash_forward(q, k, v)
print(len(res))
print("Outputs - ", res[3].shape, res[4].shape, res[5].shape, len(res))

do = jax.device_put(
    jnp.ones(shape = (4 * 8, 12, 16), dtype=jnp.float32),
    jax.sharding.NamedSharding(VMAP_MESH, P("x", None, None))
)

dq, dk, dv = flash_backward(res, do)
print("Outputs - ", dq.shape, dk.shape, dv.shape)

7
Outputs -  (32, 16, 12) (32, 12, 16) (32, 12, 1) 7
Outputs -  (32, 12, 16) (32, 12, 16) (32, 12, 16)


## Define Flash Attention VJP

In [24]:
@jax.custom_vjp
def flash_attention(q, k, v):
    o, res = flash_forward(q, k, v)
    return o

flash_attention.defvjp(flash_forward, flash_backward)

## Residual Block

In [25]:
class ResidualBlock(nnx.Module):
    def __init__(self, rngs: nnx.Rngs):
        self.attention_layer = FlashAttention(rngs = rngs)
        
        self.dropout_1 = nnx.Dropout(rate = config.model.dropout_rate, rngs = rngs)
        self.dropout_2 = nnx.Dropout(rate = config.model.dropout_rate, rngs = rngs)

        self.dense_1 = nnx.Linear(
            in_features = config.model.d,
            out_features = config.model.d,
            use_bias=True,
            kernel_init=nnx.with_partitioning(
                kernel_init_fn,
                (
                    "x" if config.model.d % devices.shape[0] == 0 else None,
                    "y" if config.model.d % devices.shape[1] == 0 else None,
                ),
                mesh = MESH
            ),
            bias_init=nnx.with_partitioning(
                bias_init_fn,
                ("x" if config.model.d % devices.shape[0] == 0 else None,),
                mesh = MESH
            ), 
            rngs = rngs
        )

        self.dense_2 = nnx.Linear(
            in_features = config.model.d,
            out_features = config.model.d,
            use_bias=True,
            kernel_init=nnx.with_partitioning(
                kernel_init_fn,
                (
                    "x" if config.model.d % devices.shape[0] == 0 else None,
                    "y" if config.model.d % devices.shape[1] == 0 else None,
                ),
                mesh = MESH
            ),
            bias_init=nnx.with_partitioning(
                bias_init_fn,
                ("x" if config.model.d % devices.shape[0] == 0 else None,),
                mesh = MESH
            ), 
            rngs = rngs
        )
        
        self.layernorm_1 = nnx.LayerNorm(
            num_features = config.model.d,
            epsilon=config.model.ln_epsilon,
            use_bias=True, use_scale=True,
            scale_init = nnx.with_partitioning(
                scale_init_fn,
                ("x" if config.model.d % devices.shape[0] == 0 else None,),
                mesh = MESH
            ),
            bias_init = nnx.with_partitioning(
                bias_init_fn,
                ("x" if config.model.d % devices.shape[0] == 0 else None,),
                mesh = MESH
            ),
            rngs = rngs
        )

        self.layernorm_2 = nnx.LayerNorm(
            num_features = config.model.d,
            epsilon=config.model.ln_epsilon,
            use_bias=True, use_scale=True,
            scale_init = nnx.with_partitioning(
                scale_init_fn,
                ("x" if config.model.d % devices.shape[0] == 0 else None,),
                mesh = MESH
            ),
            bias_init = nnx.with_partitioning(
                bias_init_fn,
                ("x" if config.model.d % devices.shape[0] == 0 else None,),
                mesh = MESH
            ),
            rngs = rngs
        )

    def __call__(self, x: jnp.ndarray):
        out = self.attention_layer(x)
        out = self.dropout_1(out)
        out = self.layernorm_1(x + out)

        ffn_out = self.dense_1(out)
        ffn_out = nnx.relu(ffn_out)

        ffn_out = self.dense_2(ffn_out)
        ffn_out = self.dropout_2(ffn_out)
        return self.layernorm_2(out + ffn_out)

    def prepare(self, x: jnp.ndarray):
        out = self.attention_layer.prepare(x)
        out = self.dropout_1(out)
        out = self.layernorm_1(x + out)

        ffn_out = self.dense_1(out)
        ffn_out = nnx.relu(ffn_out)

        ffn_out = self.dense_2(ffn_out)
        ffn_out = self.dropout_2(ffn_out)
        return self.layernorm_2(out + ffn_out)

    def inference(self, x: jnp.ndarray):
        out = self.attention_layer.inference(x)
        out = self.dropout_1(out)
        out = self.layernorm_1(x + out)

        ffn_out = self.dense_1(out)
        ffn_out = nnx.relu(ffn_out)

        ffn_out = self.dense_2(ffn_out)
        ffn_out = self.dropout_2(ffn_out)
        return self.layernorm_2(out + ffn_out)

## Sparse Transformer

In [44]:
class TinyGPT(nnx.Module):
    def __init__(self, rngs: nnx.Rngs):
        self.embeddings = Embeddings(rngs = rngs)

        # Found this from nnx.scan docs-----------------------
        # RematBlock = nnx.remat(
        #     ResidualBlock,
        #     policy=jax.checkpoint_policies.nothing_saveable
        # )
        #-----------------------------------------------------
        
        @nnx.split_rngs(splits=config.model.num_resids)
        @nnx.vmap(in_axes=(0,), out_axes=0, transform_metadata={nnx.PARTITION_NAME: None})
        def create_block(rngs):
            return ResidualBlock(rngs)
        self.resblocks = create_block(rngs)

        # self.resblocks = nnx.List([])
        # for i in range(config.model.num_resids):
        #     self.resblocks.append(ResidualBlock(rngs))
        print("All Residual Blocks Added")
        
        self.Wout = nnx.Linear(
            in_features = config.model.d,
            out_features = config.data.vocab_size,
            use_bias=False,
            kernel_init=nnx.with_partitioning(
                kernel_init_fn,
                (
                    "x" if config.model.d % devices.shape[0] == 0 else None,
                    "y" if config.data.vocab_size % devices.shape[1] == 0 else None
                ),
                mesh = MESH
            ),
            bias_init=nnx.with_partitioning(
                bias_init_fn,
                (
                    "x" if config.data.vocab_size % devices.shape[0] == 0 else None, 
                ),
                mesh = MESH
            ), 
            rngs = rngs
        )
        
    def __call__(self, x: jnp.ndarray):
        out = self.embeddings(x)
        @nnx.scan(in_axes=(nnx.Carry, 0), out_axes=nnx.Carry)
        def forward(x, model):
            return model(x)
        out = forward(out, self.resblocks)
        # for resblock in self.resblocks:
        #     out = resblock(out)
        out = self.Wout(out)
        return out

    def prepare(self, x: jnp.ndarray):
        out = self.embeddings(x)
        @nnx.scan(in_axes=(nnx.Carry, 0), out_axes=nnx.Carry)
        def forward(x, model):
            return model.prepare(x)
        out = forward(out, self.resblocks)
        # for resblock in self.resblocks:
        #     out = resblock(out)
        out = self.Wout(out)
        return out

    def inference(self, x: jnp.ndarray, position_offset=0):
        out = self.embeddings(x, position_offset=self.resblocks.attention_layer.cache.pos[...][0])
        @nnx.scan(in_axes=(nnx.Carry, 0), out_axes=nnx.Carry)
        def forward(x, model):
            return model.inference(x)
        out = forward(out, self.resblocks)
        # for resblock in self.resblocks:
        #     out = resblock(out)
        out = self.Wout(out)
        return out

## Model Creation Function

In [32]:
# @jax.jit
def create_sharded_model():
    model = TinyGPT(
        rngs = nnx.Rngs(config.model.seed)
    )
    opt = optax.adam(learning_rate=config.training.lr)
    optimizer = nnx.Optimizer(model, opt, wrt=nnx.Param)
    
    model_state = nnx.state(model, nnx.Param, nnx.Variable)
    
    param_shardings = nnx.get_named_sharding(model_state[0], MESH)
    var_shardings = nnx.get_named_sharding(model_state[1], MESH)
    
    param_sharded_state = jax.lax.with_sharding_constraint(model_state[0], param_shardings)
    var_sharded_state = jax.lax.with_sharding_constraint(model_state[1], var_shardings)
    
    nnx.update(model, param_sharded_state, var_sharded_state)

    optimizer_state = nnx.state(optimizer, nnx.optimizer.OptState)
    optimizer_shardings = nnx.get_named_sharding(optimizer_state, MESH)
    optimizer_sharded_state = jax.lax.with_sharding_constraint(optimizer_state, optimizer_shardings)
    nnx.update(optimizer, optimizer_sharded_state)
    return model, optimizer

## Loss Function

In [33]:
def loss_fn(
    model: TinyGPT,
    batch: Sequence[jnp.ndarray],
):
    logits = model(batch[0])
    loss = optax.softmax_cross_entropy_with_integer_labels(logits, batch[1]).mean()
    return loss

grad_loss_fn = nnx.value_and_grad(loss_fn)

## Tain and Eval Steps

In [34]:
@nnx.jit
def train_step(
    model: TinyGPT,
    optimizer: nnx.Optimizer,
    metrics: nnx.MultiMetric,
    batch: Sequence[jnp.ndarray],
):
    loss, grads = grad_loss_fn(model, batch)
    optimizer.update(model, grads)
    # optimizer.update(grads)
    metrics.update(loss=loss)

@nnx.jit
def eval_step(
    model: TinyGPT,
    metrics: nnx.MultiMetric,
    batch: Sequence[np.ndarray],
):  
    loss = loss_fn(model, batch)
    metrics.update(loss=loss)

## Checkpoint Manager

In [35]:
from datetime import datetime

class ManageCheckpoints:
    def __init__(
        self,
        base_dir: str,
    ):
        self.ckpt_dir = ocp.test_utils.erase_and_create_empty(base_dir)
        self.checkpointer = ocp.StandardCheckpointer()

    def save_model(
        self,
        model: nnx.Module
    ) -> None:
        _, state = nnx.split(model)
        
        new_dir = self.ckpt_dir / f"{str(int(datetime.utcnow().timestamp()))}_state"
        self.checkpointer.save(new_dir, state)
        
        return str(new_dir)
        
    def load_model(
        self,
        model: nnx.Module,
        checkpoint: str,
        mesh: jax.sharding.Mesh = None
    ) -> nnx.Module:
        abstract_model = nnx.eval_shape(lambda: model)
        graphdef, abstract_state = nnx.split(abstract_model)
        nnx.display(abstract_state)

        if mesh:
            abstract_state = jax.tree.map(
                lambda a, s: jax.ShapeDtypeStruct(a.shape, a.dtype, sharding=s),
                abstract_state,
                nnx.get_named_sharding(abstract_state, mesh)
            )
        
        restored_state = self.checkpointer.restore(self.ckpt_dir / checkpoint, abstract_state)
        nnx.display(restored_state)
        
        return nnx.merge(graphdef, restored_state)

## Train Definition

In [36]:
def train(
    epochs: int,

    model: TinyGPT,
    optimizer: nnx.Optimizer,
    metrics: nnx.MultiMetric,
    
    train_dataset: grain.MapDataset,
    val_dataset: grain.MapDataset,

    train_metrics: dict,
    val_metrics: dict,

    sample_dir: str,
    checkpoint_manager: ManageCheckpoints = None,
):
    table = wandb.Table(
        columns=["actual", "completions",],
        log_mode="MUTABLE"
    )
    
    train_key = jax.random.key(config.model.seed)
    
    for epoch in range(1, epochs + 1):
        # ----------------------------
        # Training
        # ----------------------------
        model.train()
        with tqdm.tqdm(train_dataset, unit="batch") as train_epochs:
            for index, batch in enumerate(train_epochs):
                step = len(train_dataset) * (epoch - 1) + index
                
                train_epochs.set_description(f"Epoch {epoch} | Training | ")
                
                train_step(
                    model=model,
                    optimizer=optimizer,
                    metrics=metrics,
                    batch=batch
                )

                # gather batch metrics
                batch_metrics = metrics.compute()
                for metric, value in batch_metrics.items():
                    train_metrics[metric].append(value)
                metrics.reset()

                run.log({"train_loss": batch_metrics["loss"]})
                train_epochs.set_postfix(**{f"{k}": v for k, v in batch_metrics.items()},)

                # ----------------------------
                # Checkpointing
                # ----------------------------
                if (step + 1) % config.training.save_and_sample_every == 0:
                # if (step + 1) % 10 == 0:
                    checkpoint_dir = checkpoint_manager.save_model(model)
                    
                    max_retries = 30
                    while not os.path.isdir(checkpoint_dir) and max_retries > 0:
                        print("Waiting for Checkpoint manager")
                        time.sleep(15)
                        max_retries -= 1
                    
                    if os.path.isdir(checkpoint_dir):
                        artifact = wandb.Artifact(name=f"model_{step + 1}", type="sparse_transformer")
                        artifact.add_dir(checkpoint_dir)
                        run.log_artifact(artifact)
                        print(f"Saved checkpoint at {checkpoint_dir}")

                    #SAMPLING SKIPPED SINCE SLOW -------------------------------
                    train_key, subkey = jax.random.split(train_key)
                    index = jax.random.randint(subkey, shape=(), minval=0, maxval=len(val_dataset))
                    sample_batch = val_dataset[index]
                    sample_arrs = jax.block_until_ready(sample(model, sample_batch))
                    host_arrs = jax.device_get(sample_arrs)
                    for i in range(config.sampling.batch_size):
                        table.add_data(
                            tokenizer.decode(sample_batch[0][i, ...]),
                            tokenizer.decode(host_arrs[i, ...])
                        )
                    run.log({"completions_table": table})
                    #------------------------------------------------------------
        
        checkpoint_dir = checkpoint_manager.save_model(model)
        max_retries = 30
        while not os.path.isdir(checkpoint_dir) and max_retries > 0:
            print("Waiting for Checkpoint manager")
            time.sleep(15)
            max_retries -= 1
        if os.path.isdir(checkpoint_dir):
            artifact = wandb.Artifact(name=f"model_{step + 1}", type="sparse_transformer")
            artifact.add_dir(checkpoint_dir)
            run.log_artifact(artifact)
            print(f"Saved checkpoint at {checkpoint_dir}")

        # ----------------------------
        # Validation
        # ----------------------------
        model.eval()
        with tqdm.tqdm(val_dataset[:2000], unit="batch") as val_epochs:
            for index, batch in enumerate(val_epochs):
                val_epochs.set_description(f"Epoch {epoch} | Validation | ")

                eval_step(
                    model=model,
                    metrics=metrics,
                    batch=batch,
                )

                # gather batch metrics
                batch_metrics = metrics.compute()
                for metric, value in batch_metrics.items():
                    val_metrics[metric].append(value)
                metrics.reset()

                run.log({"val_loss": batch_metrics["loss"]})
                val_epochs.set_postfix(**{f"{k}": v for k, v in batch_metrics.items()},)

        # ----------------------------
        # Epoch Summary
        # ----------------------------
        print(f"\nEpoch {epoch} Summary:")
        for metric in train_metrics.keys():
            train_avg = np.mean(train_metrics[metric][-len(train_dataset):])
            val_avg = np.mean(val_metrics[metric][-len(val_dataset):])
            print(f"Model {metric} | Train: {train_avg:.4f} | Val: {val_avg:.4f}")
            
    return train_metrics, val_metrics

## Creation

In [37]:
if not os.path.exists("/kaggle/working/checkpoints"):
    os.mkdir("/kaggle/working/checkpoints")
    print("/kaggle/working/checkpoints")

if not os.path.exists("/kaggle/working/samples"):
    os.mkdir("/kaggle/working/samples")
    print("/kaggle/working/samples")

/kaggle/working/checkpoints
/kaggle/working/samples


In [72]:
try:
    del model
    del optimizer
    del metrics
    del checkpoint_manager
except NameError:
    pass

print(gc.collect(), jax.clear_caches())

1389 None


In [73]:
model, optimizer = create_sharded_model()
checkpoint_manager = ManageCheckpoints(base_dir = "/kaggle/working/checkpoints")
metrics = nnx.MultiMetric(loss=nnx.metrics.Average('loss'))

All Residual Blocks Added


In [74]:
nnx.display(model, optimizer)

## Sampling

In [75]:
# api = wandb.Api()  # requires WANDB_API_KEY env var
# artifact = api.artifact("YOUR_ENTITY/jax-flash-minigpt/model_STEP:v0")
# artifact_dir = artifact.download(root="/kaggle/working/checkpoints/model_STEP")
# model = checkpoint_manager.load_model(model=model, checkpoint="model_STEP", mesh=MESH)

wandb: Downloading large artifact 'model_17662:v0', 97.48MB. 11 files...
wandb:   11 of 11 files downloaded.  
Done. 00:00:00.5 (203.7MB/s)


In [87]:
@nnx.jit
def sample_token(logits, rng):
    logits = logits / (config.sampling.temperature + 1e-8)
    # top_k_vals = jax.lax.dynamic_index_in_dim(
    #     jax.lax.sort(logits, dimension = -1),
    #     logits.shape[2] - config.sampling.top_k,
    #     axis = 2,
    #     keepdims = True
    # )
    # logits = jnp.where(logits >= top_k_vals, logits, -jnp.inf)
    # next_token = jax.random.categorical(rng, logits, axis=-1)  # (B, 1)
    # return next_token

    keys = jax.random.split(jax.random.PRNGKey(0), logits.shape[0])
    logits, indices = jax.lax.top_k(logits, k=config.sampling.top_k)
    logits = nnx.softmax(logits)
    tokens = [
        jax.random.choice(
            keys[i],
            indices[i, 0],
            p=logits[i, 0]
        ) for i in range(logits.shape[0])
    ]
    tokens = jnp.array(tokens)[:, None]
    return tokens

@nnx.jit
def prepare(model, x):
    return model.prepare(x)

@nnx.jit
def sample_step(model, x):
    return model.inference(x)

@nnx.jit(static_argnums = (1,))
def prepare_initial(batch, length=20):
    return jax.lax.dynamic_slice_in_dim(batch[0], 0, length, axis=1)

def sample(model, batch, seed=42):
    model.eval()
    rng = jax.random.PRNGKey(seed)

    length = 20
    T = config.model.max_len

    x = prepare_initial(batch, length=length)
    logits = prepare(model, x)
    last_logit = jax.lax.dynamic_index_in_dim(logits, logits.shape[1] - 1, axis=1, keepdims=True)
    
    rng, subrng = jax.random.split(rng)
    first_gen_token = sample_token(last_logit, subrng)
    length += 1

    rng, loop_rng = jax.random.split(rng)
    scan_rngs = jax.random.split(loop_rng, T - length)
    
    # @nnx.scan(in_axes=(nnx.Carry, 0), out_axes=(nnx.Carry, 1))
    # def decode_step(carries, step_rng):
    #     model, current_token, current_step = carries
    #     logit = sample_step(model, current_token)
    #     next_token = sample_token(logit, step_rng)
        
    #     return (model, next_token, current_step + 1), jax.lax.dynamic_index_in_dim(next_token, 0, axis=1, keepdims=False)
    # (_, _, _), generated_tokens = decode_step((model, first_gen_token, length), scan_rngs)
    # return jnp.concatenate([x, first_gen_token, generated_tokens], axis=1)

    generated = []
    current_token = first_gen_token
    for lrngs in scan_rngs:
        logit = sample_step(model, current_token)
        current_token = sample_token(logit, lrngs)
        generated.append(current_token)
        length += 1

    model.train()
    return jnp.concatenate([x, first_gen_token] + generated, axis=1)
    

In [88]:
idx = 1000
sample_batch = val_dataset[idx]
sample_arr = jax.block_until_ready(sample(model, sample_batch))
sample_arr.shape

(16, 257)

In [91]:
tokenizer.decode(sample_arr[5, ...])

'Once upon a time, there was a wise old man who lived on a street. He knew how to make a big difference. One day, he decided to make a big cake. He mixed the ingredients together and put it in the oven.\n\nThe wise old man was very proud of his cake. He wanted to show it to everyone. He took out his cake and put it on the oven. Everyone was amazed at how delicious it looked.\n\nThe wise old man was very proud of his cake. He showed it to everyone. Everyone was amazed at how delicious it looked.\n\nThe wise old man was very proud of his cake. He knew that it was a special cake and that everyone was very happy.<|endoftext|>!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!'

## Training

### Single Batch Test

In [42]:
train_metrics, val_metrics = defaultdict(list), defaultdict(list)

train_metrics, val_metrics = train(
    epochs = config.training.epochs,

    model = model,
    optimizer = optimizer,
    metrics = metrics,
    
    train_dataset = train_dataset,
    val_dataset = val_dataset,

    train_metrics = train_metrics,
    val_metrics = val_metrics,

    checkpoint_manager = checkpoint_manager,
    sample_dir = "/kaggle/working/samples",
)

Epoch 1 | Training | :   6%|▌         | 999/17662 [13:01<3:29:05,  1.33batch/s, loss=1.796368] /tmp/ipykernel_58/3655167565.py:17: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  new_dir = self.ckpt_dir / f"{str(int(datetime.utcnow().timestamp()))}_state"


Waiting for Checkpoint manager


wandb: Adding directory to artifact (/kaggle/working/checkpoints/1785169313_state)... Done. 0.4s


Saved checkpoint at /kaggle/working/checkpoints/1785169313_state


Epoch 1 | Training | :  11%|█▏        | 1999/17662 [26:45<3:17:05,  1.32batch/s, loss=1.5666555] 

...

Waiting for Checkpoint manager


wandb: Adding directory to artifact (/kaggle/working/checkpoints/1785182203_state)... Done. 0.2s


Saved checkpoint at /kaggle/working/checkpoints/1785182203_state


Epoch 1 | Validation | : 100%|██████████| 1374/1374 [01:07<00:00, 20.21batch/s, loss=1.2318506]



Epoch 1 Summary:
Model loss | Train: 1.4948 | Val: 1.2882


In [ ]:
run.finish()

In [ ]:
# import os
# import tarfile
# from IPython.display import FileLink

# # Define your source and output paths
# source_dir = '/kaggle/working/xla_debug' # Change to your folder name
# output_filename = 'xla_debug.tar.gz'

# # 1. Compress the directory
# with tarfile.open(output_filename, "w:gz") as tar:
#     tar.add(source_dir, arcname=os.path.basename(source_dir))

# print(f"Successfully created {output_filename}")

# # 2. Generate the download link
# FileLink(output_filename)